# Week 7: Pre-Class Practice

**Topic:** Diagnosing Overfitting - Detective Training  
**Estimated Time:** 10-15 minutes  
**Scaffolding:** HIGH (focus on observation & diagnosis)  
**Goal:** Learn to spot overfitting by comparing three models

---

## Why This Matters

In Week 7, you'll learn how to **fix** overfitting in production models. But first, you need to **recognize** it when you see it!

This notebook is your "detective training" - you'll see three models with different problems and learn to diagnose them by reading their learning curves.

---

## Learning Objectives

1. Recognize overfitting from learning curves (training vs validation gap)
2. Distinguish overfitting from underfitting
3. Identify a well-fit model
4. Preview solutions (Dropout, EarlyStopping) for live session

---

**Note:** We're using a simple 2D dataset here so patterns are obvious. In the live session, you'll apply these same skills to real Fashion-MNIST images!

---

## Part 1: Setup & Simple Dataset

We'll use a simple 2D "moons" dataset - two interleaving half-circles. This is perfect for learning because:
- It's visual (you can plot it)
- It's simple (trains in seconds)
- Overfitting patterns are obvious

In [ ]:
# Setup
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

print(f"Keras version: {keras.__version__}")
print("Ready for detective work!")

In [ ]:
# Create simple 2D dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Feature dimensions: {X_train.shape[1]}")

# Visualize the data
plt.figure(figsize=(8, 6))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], 
            c='blue', label='Class 0', alpha=0.6, edgecolor='k')
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], 
            c='red', label='Class 1', alpha=0.6, edgecolor='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Two Moons Dataset - Binary Classification')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n👀 Notice: Two interleaving half-circles - a classic classification challenge!")

---

## Part 2: Model A - "The Eager Student" 🤓

**Hypothesis:** This model is TOO complex and will overfit (memorize training data)

**Architecture:**
- 3 Dense layers
- 128 neurons each
- Training for 200 epochs

**Watch for:** Gap between training and validation curves

In [ ]:
# Build Model A: The Eager Student (Complex network)
from keras import layers

model_A = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name="Model_A_Eager")

model_A.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model A: The Eager Student")
model_A.summary()

In [ ]:
# Train Model A
print("Training Model A (The Eager Student)...")
history_A = model_A.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0  # Silent training
)

print("✅ Training complete!")
print(f"Final Training Accuracy: {history_A.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history_A.history['val_accuracy'][-1]:.4f}")
print(f"Gap: {(history_A.history['accuracy'][-1] - history_A.history['val_accuracy'][-1]):.4f}")

In [ ]:
# Plot Model A curves
plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(history_A.history['loss'], label='Training Loss', linewidth=2, color='blue')
plt.plot(history_A.history['val_loss'], label='Validation Loss', linewidth=2, color='orange')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=11)
plt.title('Model A: Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(history_A.history['accuracy'], label='Training Accuracy', linewidth=2, color='blue')
plt.plot(history_A.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='orange')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.title('Model A: Accuracy Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 🔍 Detective Questions - Model A

Look at the curves above and answer:

1. **Is this model overfitting?** (Look for a BIG gap between blue and orange lines)
2. **What clues tell you this?** (Hint: Does training accuracy keep improving while validation plateaus or gets worse?)
3. **Would you deploy this to production?** (Would it work well on new, unseen data?)

**Think about it before moving on!**

<details>
<summary>Click to see diagnosis</summary>

**Diagnosis: OVERFITTING** 🔴

**Evidence:**
- Training accuracy → very high (near 100%)
- Validation accuracy → much lower and plateaus
- Large gap between training and validation curves
- Model is "memorizing" training data rather than learning general patterns

**Like a student who memorizes answers but doesn't understand concepts - fails on new questions!**

**Deploy to production?** ❌ NO - it won't generalize to new data

</details>

---

## Part 3: Model B - "The Lazy Student" 😴

**Hypothesis:** This model is TOO simple and will underfit (can't learn patterns)

**Architecture:**
- 2 Dense layers
- Only 4 neurons each
- Training for 50 epochs

**Watch for:** Both training AND validation accuracy stay low

In [ ]:
# Build Model B: The Lazy Student (Too simple)
model_B = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(4, activation='relu'),
    layers.Dense(4, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name="Model_B_Lazy")

model_B.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model B: The Lazy Student")
model_B.summary()

In [ ]:
# Train Model B
print("Training Model B (The Lazy Student)...")
history_B = model_B.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)

print("✅ Training complete!")
print(f"Final Training Accuracy: {history_B.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history_B.history['val_accuracy'][-1]:.4f}")
print(f"Gap: {(history_B.history['accuracy'][-1] - history_B.history['val_accuracy'][-1]):.4f}")

In [ ]:
# Plot Model B curves
plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(history_B.history['loss'], label='Training Loss', linewidth=2, color='blue')
plt.plot(history_B.history['val_loss'], label='Validation Loss', linewidth=2, color='orange')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=11)
plt.title('Model B: Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(history_B.history['accuracy'], label='Training Accuracy', linewidth=2, color='blue')
plt.plot(history_B.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='orange')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.title('Model B: Accuracy Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 🔍 Detective Questions - Model B

Look at the curves above and answer:

1. **Is this model overfitting?** (Is there a big gap?)
2. **Is this model underfitting?** (Are BOTH accuracies low?)
3. **Would you deploy this to production?**

**Think about it before moving on!**

<details>
<summary>Click to see diagnosis</summary>

**Diagnosis: UNDERFITTING** 🔵

**Evidence:**
- Training accuracy → LOW (~75%)
- Validation accuracy → LOW (~73%)
- Small gap (model is consistent)
- But performance is bad on BOTH sets!

**Like a student who doesn't study enough - performs poorly on everything!**

**Deploy to production?** ❌ NO - accuracy is too low to be useful

</details>

---

## Part 4: Model C - "The Goldilocks" ⭐

**Hypothesis:** This model is JUST RIGHT - good fit!

**Architecture:**
- 2 Dense layers
- 16 neurons each (not too big, not too small)
- Training for 100 epochs

**Watch for:** Small gap AND high performance on both training and validation

In [ ]:
# Build Model C: The Goldilocks (Just right!)
model_C = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name="Model_C_Goldilocks")

model_C.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model C: The Goldilocks")
model_C.summary()

In [ ]:
# Train Model C
print("Training Model C (The Goldilocks)...")
history_C = model_C.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)

print("✅ Training complete!")
print(f"Final Training Accuracy: {history_C.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history_C.history['val_accuracy'][-1]:.4f}")
print(f"Gap: {(history_C.history['accuracy'][-1] - history_C.history['val_accuracy'][-1]):.4f}")

In [ ]:
# Plot Model C curves
plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(history_C.history['loss'], label='Training Loss', linewidth=2, color='blue')
plt.plot(history_C.history['val_loss'], label='Validation Loss', linewidth=2, color='orange')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=11)
plt.title('Model C: Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(history_C.history['accuracy'], label='Training Accuracy', linewidth=2, color='blue')
plt.plot(history_C.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='orange')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.title('Model C: Accuracy Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 🔍 Detective Questions - Model C

Look at the curves above and answer:

1. **Is this model overfitting?** (Check the gap)
2. **Is this model underfitting?** (Check if accuracies are high)
3. **Would you deploy this to production?**

**Think about it before moving on!**

<details>
<summary>Click to see diagnosis</summary>

**Diagnosis: GOOD FIT** 🟢

**Evidence:**
- Training accuracy → HIGH (~90%)
- Validation accuracy → HIGH (~88%)
- Small gap (~2% difference)
- Model learns general patterns that transfer to new data

**Like a student who studies well - understands concepts and applies them to new problems!**

**Deploy to production?** ✅ YES - good balance of performance and generalization

</details>

---

## Part 5: Side-by-Side Comparison

Now let's see all three models together. This is the power of comparative analysis!

**Can you spot the differences at a glance?**

---

## Summary: What You Learned

### 🎯 Core Skills Acquired

1. **Reading Learning Curves**
   - Blue line = training performance
   - Orange line = validation performance
   - Gap between them = key diagnostic signal

2. **Diagnosing Overfitting**
   - Large gap = overfitting (memorization)
   - Training keeps improving, validation plateaus/drops
   - Like the "Eager Student" who memorizes answers

3. **Diagnosing Underfitting**
   - Small gap, but both curves LOW
   - Model too simple to learn patterns
   - Like the "Lazy Student" who doesn't study

4. **Recognizing Good Fit**
   - Small gap, both curves HIGH
   - Balanced model that generalizes
   - Like the "Goldilocks" - just right!

---

## What's Next: Week 7 Live Session

**You'll apply these skills to real Fashion-MNIST images:**

1. **Intentionally create overfitting** (like Model A)
2. **Diagnose it** using the skills you learned today
3. **Fix it** with Dropout, EarlyStopping, and regularization
4. **Build production-ready classifier** with proper validation
5. **Pair programming** - diagnose and fix partner's overfit model

**You're prepared!** You know what overfitting LOOKS like, so when you see it in Fashion-MNIST, you'll recognize it immediately.

---

## Key Metaphors to Remember

- 🤓 **"The Eager Student"** = Overfitting (memorizes, doesn't generalize)
- 😴 **"The Lazy Student"** = Underfitting (doesn't learn enough)
- ⭐ **"The Goldilocks"** = Good Fit (balanced, just right)

---

**Great work, detective! See you in Week 7 live session!** 🎉

---

*Week 7 Pre-Class Practice | Redesigned Version 2.0 | March 2026*

### Question 3

Which technique would you try FIRST to fix overfitting?

A) Add more layers  
B) Train for more epochs  
C) Add Dropout layers  
D) Make the network bigger  

<details>
<summary>Click for answer</summary>

**Answer: C) Add Dropout layers**

**Why:** Dropout is a direct solution to overfitting - it prevents memorization by randomly disabling neurons. Options A, B, and D would make overfitting WORSE!

</details>

---

### Question 2

You see these curves:
- Training accuracy: 68%
- Validation accuracy: 66%
- Small gap, both curves plateau early

**What's the problem?**

A) Underfitting  
B) Overfitting  
C) Good fit  
D) Need fewer epochs  

<details>
<summary>Click for answer</summary>

**Answer: A) Underfitting**

**Why:** Small gap is good, but BOTH accuracies are low. Model is too simple. The "Lazy Student" problem!

</details>

---

### Question 1

You see these curves for a new model:
- Training accuracy: 95%
- Validation accuracy: 78%
- Large gap that grows over time

**What's the problem?**

A) Underfitting  
B) Overfitting  
C) Good fit  
D) Need more data  

<details>
<summary>Click for answer</summary>

**Answer: B) Overfitting**

**Why:** Large gap (95% vs 78%) means the model is memorizing training data. The "Eager Student" problem!

</details>

---

---

## Part 7: Test Your Understanding

Before the live session, see if you can answer these questions:

In [ ]:
# PREVIEW: Solutions for overfitting (you'll implement these in class!)

print("🛠️ Technique 1: DROPOUT")
print("   Randomly 'turn off' neurons during training")
print("   Code example:")
print("   layers.Dropout(0.3)  # Turn off 30% of neurons randomly")
print()

print("🛠️ Technique 2: EARLY STOPPING")
print("   Stop training when validation stops improving")
print("   Code example:")
print("   keras.callbacks.EarlyStopping(patience=10)")
print()

print("🛠️ Technique 3: SIMPLER ARCHITECTURE")
print("   Use fewer layers or neurons")
print("   Example: 128 → 32 neurons")
print()

print("🛠️ Technique 4: REGULARIZATION")
print("   Add penalty for complex models")
print("   Code example:")
print("   layers.Dense(64, kernel_regularizer='l2')")
print()

print("📚 In Week 7 live session, you'll:")
print("   ✓ Implement all these techniques on Fashion-MNIST")
print("   ✓ See how they reduce the gap")
print("   ✓ Build production-ready classifiers")
print("   ✓ Practice pair programming to fix overfitting")

## Part 6: How Do We Fix Overfitting? (Preview)

You've learned to DIAGNOSE overfitting. In the live session, you'll learn to FIX it!

Here's a preview of the techniques you'll use with Fashion-MNIST:

### 💡 What You Should Notice

**Model A (Overfit):**
- Training curve keeps climbing
- Validation curve plateaus or drops
- Large gap = memorizing, not learning

**Model B (Underfit):**
- Both curves stay low
- Small gap but bad performance
- Too simple to capture patterns

**Model C (Good Fit):**
- Both curves climb together
- Small gap at the end
- Balanced complexity

**Question:** Which model would you choose for production? Why?

---

In [ ]:
# Side-by-side comparison of all three models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Model A - The Eager Student (Overfit)
axes[0].plot(history_A.history['accuracy'], label='Training', linewidth=2, color='blue')
axes[0].plot(history_A.history['val_accuracy'], label='Validation', linewidth=2, color='orange')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model A: "The Eager Student"\n(OVERFITTING 🔴)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.5, 1.0])

# Model B - The Lazy Student (Underfit)
axes[1].plot(history_B.history['accuracy'], label='Training', linewidth=2, color='blue')
axes[1].plot(history_B.history['val_accuracy'], label='Validation', linewidth=2, color='orange')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Model B: "The Lazy Student"\n(UNDERFITTING 🔵)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0.5, 1.0])

# Model C - The Goldilocks (Good fit)
axes[2].plot(history_C.history['accuracy'], label='Training', linewidth=2, color='blue')
axes[2].plot(history_C.history['val_accuracy'], label='Validation', linewidth=2, color='orange')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Accuracy', fontsize=12)
axes[2].set_title('Model C: "The Goldilocks"\n(GOOD FIT 🟢)', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0.5, 1.0])

plt.tight_layout()
plt.show()

print("\n🎯 Key Patterns to Remember:")
print("  • OVERFITTING: Big gap (training ↑↑, validation plateaus)")
print("  • UNDERFITTING: No gap, but both LOW")
print("  • GOOD FIT: Small gap, both HIGH")